# Customer Churn Prediction
Clean, fixed version — EDA → Preprocessing → Logistic Regression → KNN → SVM → Random Forest

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('customer_churn_data.csv')
df.head()

## 2. Explore Data

In [ ]:
df.info()
print('\nShape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicates:', df.duplicated().sum())

In [ ]:
df.describe()

## 3. Clean Data

In [ ]:
df['InternetService'] = df['InternetService'].fillna('')
print('Missing values after fix:', df.isna().sum().sum())

## 4. EDA — Charts

In [ ]:
df['Churn'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Churn (Yes/No)')
plt.ylabel('')
plt.show()
print('Churn balance (normalized):')
print(df['Churn'].value_counts(normalize=True))

In [ ]:
print('Mean MonthlyCharges by Churn:')
print(df.groupby('Churn')['MonthlyCharges'].mean())
print('\nMean MonthlyCharges by Churn + Gender:')
print(df.groupby(['Churn', 'Gender'])['MonthlyCharges'].mean())
print('\nMean Tenure by Churn:')
print(df.groupby('Churn')['Tenure'].mean())
print('\nMean Age by Churn:')
print(df.groupby('Churn')['Age'].mean())

In [ ]:
df.groupby('ContractType')['MonthlyCharges'].mean().plot(kind='bar')
plt.ylabel('Mean Price')
plt.xlabel('Contract Type')
plt.title('Contract Type Average Price')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['MonthlyCharges'], bins=20)
axes[0].set_title('Histogram of Monthly Charges')
axes[1].hist(df['Tenure'], bins=20)
axes[1].set_title('Histogram of Tenure')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=['number'])
print('Correlation matrix:')
numeric_cols.corr()

## 5. Preprocessing
> Encode FIRST, split ONCE, scale ONCE. Never split then re-encode.

In [ ]:
# Encode Gender and Churn BEFORE splitting
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

X = df[['Age', 'Gender', 'Tenure', 'MonthlyCharges']]
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)  # Series (1D), not DataFrame

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nClass balance:')
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Split ONCE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale ONCE — fit on train only, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'scaler.pkl')
print('Train size:', X_train_scaled.shape)
print('Test size:', X_test_scaled.shape)

In [ ]:
print('NaNs in X_train:', np.isnan(X_train_scaled).sum())
print('NaNs in y_train:', np.isnan(y_train).sum())

## 6. Helper — Model Performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def model_performance(model_name, predictions):
    print(f'=== {model_name} ===')
    print(f'Accuracy: {accuracy_score(y_test, predictions):.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, predictions))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, predictions))
    print()

## 7. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)

y_pred_lr = log_model.predict(X_test_scaled)
model_performance('Logistic Regression', y_pred_lr)

## 8. KNN with GridSearchCV

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],  # FIXED: was 'unifrom'
}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)

print('Best KNN params:', grid_knn.best_params_)
y_pred_knn = grid_knn.predict(X_test_scaled)
model_performance('KNN', y_pred_knn)

## 9. SVM with GridSearchCV

In [ ]:
from sklearn.svm import SVC

param_grid_svm = {
    'C': [0.01, 0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly']
}

grid_svm = GridSearchCV(
    SVC(class_weight='balanced', random_state=42),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train_scaled, y_train)  # FIXED: now uses scaled data

print('Best SVM params:', grid_svm.best_params_)
y_pred_svm = grid_svm.predict(X_test_scaled)
model_performance('SVM', y_pred_svm)

## 10. Random Forest with GridSearchCV

In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train_scaled, y_train)

print('Best RF params:', grid_rf.best_params_)
y_pred_rf = grid_rf.predict(X_test_scaled)
model_performance('Random Forest', y_pred_rf)

## 11. Save Best Model

In [ ]:
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(grid_rf.best_estimator_, 'rf_model.pkl')
print('scaler.pkl saved')
print('rf_model.pkl saved')

## 12. Model Comparison Summary

In [ ]:
results = {
    'Logistic Regression': accuracy_score(y_test, y_pred_lr),
    'KNN':                  accuracy_score(y_test, y_pred_knn),
    'SVM':                  accuracy_score(y_test, y_pred_svm),
    'Random Forest':        accuracy_score(y_test, y_pred_rf),
}

pd.Series(results).sort_values(ascending=False).plot(kind='bar', color='steelblue')
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print('\nFinal Scores:')
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {name}: {score:.4f}')